In [1]:
import os
import sqlite3
import pandas as pd

YEARS = [2023, 2024, 2025]
PROCESSED_PATH = os.path.join("data", "processed")
DB_PATH = os.path.join("contracts.db")  # kept as-is: already linked to the Power BI ODBC connection

DROP_ORPHANS = False          # True deletes rows with no parent; False only reports them
PARTIES_BY_PARTY_ID = False   # True = one row per party_id, as in load_sqlite.py

COLUMN_MAP = {
    "records": {
        "ocid": "ocid",
        "compiledRelease/date": "release_date",
        "compiledRelease/buyer/id": "buyer_id",
        "compiledRelease/buyer/name": "buyer_name",
        "compiledRelease/planning/budget/amount/amount": "budget_amount",
        "compiledRelease/tender/id": "tender_id",
        "compiledRelease/tender/title": "tender_title",
        "compiledRelease/tender/procuringEntity/id": "procuring_entity_id",
        "compiledRelease/tender/procuringEntity/name": "procuring_entity_name",
        "compiledRelease/tender/datePublished": "tender_published_date",
        "compiledRelease/tender/procurementMethod": "procurement_method",
        "compiledRelease/tender/mainProcurementCategory": "procurement_category",
        "compiledRelease/tender/value/amount": "tender_value_amount",
        "compiledRelease/tender/value/currency": "tender_value_currency",
        "compiledRelease/tender/value/amount_PEN": "tender_value_amount_pen",
        "compiledRelease/tender/numberOfTenderers": "number_of_tenderers",
        "compiledRelease/tender/tenderPeriod/startDate": "tender_start_date",
        "compiledRelease/tender/tenderPeriod/endDate": "tender_end_date",
    },
    "com_awards": {
        "ocid": "ocid",
        "compiledRelease/awards/0/id": "award_id",
        "compiledRelease/awards/0/value/amount": "award_amount",
        "compiledRelease/awards/0/value/currency": "award_currency",
        "compiledRelease/awards/0/date": "award_date",
    },
    "com_awa_suppliers": {
        "ocid": "ocid",
        "compiledRelease/awards/0/id": "award_id",
        "compiledRelease/awards/0/suppliers/0/id": "supplier_id",
        "compiledRelease/awards/0/suppliers/0/name": "supplier_name",
    },
    "com_contracts": {
        "ocid": "ocid",
        "compiledRelease/contracts/0/id": "contract_id",
        "compiledRelease/contracts/0/awardID": "award_id",
        "compiledRelease/contracts/0/title": "contract_title",
        "compiledRelease/contracts/0/value/amount": "contract_amount",
        "compiledRelease/contracts/0/dateSigned": "date_signed",
        "compiledRelease/contracts/0/period/startDate": "period_start_date",
        "compiledRelease/contracts/0/period/endDate": "period_end_date",
        "compiledRelease/contracts/0/implementation/finalValue/amount": "final_value_amount",
    },
    "com_parties": {
        "ocid": "ocid",
        "compiledRelease/parties/0/id": "party_id",
        "compiledRelease/parties/0/name": "party_name",
        "compiledRelease/parties/0/identifier/id": "identifier_id",
        "compiledRelease/parties/0/identifier/scheme": "identifier_scheme",
        "compiledRelease/parties/0/roles": "roles",
        "compiledRelease/parties/0/address/region": "region",
        "compiledRelease/parties/0/address/department": "department",
    },
    "com_ten_tenderers": {
        "ocid": "ocid",
        "compiledRelease/tender/id": "tender_id",
        "compiledRelease/tender/tenderers/0/id": "tenderer_id",
        "compiledRelease/tender/tenderers/0/name": "tenderer_name",
    },
}

## Helper functions

In [2]:
NULL_VALUES = {"", "nan", "none", "null", "n/a", "na", "-", "s/n"}


def clean_text(s, upper=False):
    # collapse repeated spaces, blanks and "NaN"-like markers become null
    s = s.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.mask(s.str.lower().isin(NULL_VALUES))
    return s.str.upper() if upper else s


def clean_code(s):
    # ids as text, strip the ".0" left over from a prior float conversion
    return clean_text(s).str.replace(r"\.0$", "", regex=True)


def clean_amount(s, name=None):
    # numeric; a negative amount is invalid and becomes null
    n = pd.to_numeric(s, errors="coerce")
    negatives = (n < 0).sum()
    if name and negatives:
        print(f"  {name}: {negatives:,} negative values nulled")
    return n.mask(n < 0)


def clean_int(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")


def clean_date(s):
    # with timezone -> Lima time. Without timezone, assumed already local
    s = s.astype("string").str.strip()
    has_tz = s.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", na=False)
    tz_dates = pd.to_datetime(s.where(has_tz), errors="coerce", utc=True, format="mixed")
    tz_dates = tz_dates.dt.tz_convert("America/Lima").dt.tz_localize(None)
    local_dates = pd.to_datetime(s.where(~has_tz), errors="coerce", format="mixed")
    return tz_dates.fillna(local_dates)


def build_key(*cols):
    out = cols[0]
    for c in cols[1:]:
        out = out + "-" + c
    return out


def load_raw(table_type):
    # concatenate all years, tag the source year
    mapping = COLUMN_MAP[table_type]
    parts = []
    for year in YEARS:
        path = os.path.join(PROCESSED_PATH, f"{table_type}_{year}_completo.csv")  # filename kept as-is
        df = pd.read_csv(path, usecols=list(mapping), dtype=str).rename(columns=mapping)
        df["year"] = year
        parts.append(df)
    out = pd.concat(parts, ignore_index=True)
    out["year"] = out["year"].astype("Int16")
    print(f"{table_type}: {len(out):,} rows read")
    return out


def dedupe_table(df, pk, name, sort_by=("year",)):
    # no key -> dropped. repeated key (e.g. across years) -> keep the most recent version
    n0 = len(df)
    df = df[df[pk].notna()]
    n1 = len(df)
    df = df.sort_values(list(sort_by), na_position="first").drop_duplicates(pk, keep="last")
    print(f"{name}: {n0 - n1:,} missing key, {n1 - len(df):,} duplicated -> {len(df):,} rows")
    return df.reset_index(drop=True)

## records

In [3]:
rec = load_raw("records")

rec["ocid"] = clean_text(rec["ocid"])
for c in ["buyer_id", "tender_id", "procuring_entity_id"]:
    rec[c] = clean_code(rec[c])

for c in ["buyer_name", "procuring_entity_name"]:
    rec[c] = clean_text(rec[c], upper=True)
rec["tender_title"] = clean_text(rec["tender_title"])
rec["tender_value_currency"] = clean_text(rec["tender_value_currency"], upper=True)
for c in ["procurement_method", "procurement_category"]:
    rec[c] = clean_text(rec[c]).str.lower()

for c in ["budget_amount", "tender_value_amount"]:
    rec[c] = clean_amount(rec[c], c)

rec["tender_value_amount_pen"] = pd.to_numeric(rec["tender_value_amount_pen"], errors="coerce")
rec["number_of_tenderers"] = clean_int(rec["number_of_tenderers"])

for c in ["release_date", "tender_published_date", "tender_start_date", "tender_end_date"]:
    rec[c] = clean_date(rec[c])

# derived columns
rec["tender_duration_days"] = (rec["tender_end_date"] - rec["tender_start_date"]).dt.days.astype("Int64")
rec["tender_year"] = rec["tender_published_date"].dt.year.astype("Int16")
rec["invalid_period"] = rec["tender_duration_days"].lt(0).fillna(False)

records = dedupe_table(rec, "ocid", "records", sort_by=("year", "release_date"))
records.dtypes

records: 236,234 rows read
records: 0 missing key, 0 duplicated -> 236,234 rows


ocid                       string[python]
release_date               datetime64[ns]
buyer_id                   string[python]
buyer_name                 string[python]
budget_amount                     float64
tender_id                  string[python]
tender_title               string[python]
procuring_entity_id        string[python]
procuring_entity_name      string[python]
tender_published_date      datetime64[ns]
procurement_method         string[python]
procurement_category       string[python]
tender_value_amount               float64
tender_value_currency      string[python]
tender_value_amount_pen           float64
tender_start_date          datetime64[ns]
tender_end_date            datetime64[ns]
number_of_tenderers                 Int64
year                                Int16
tender_duration_days                Int64
tender_year                         Int16
invalid_period                    boolean
dtype: object

## awards

In [4]:
aw = load_raw("com_awards")

aw["ocid"] = clean_text(aw["ocid"])
aw["award_id"] = clean_code(aw["award_id"])
aw["award_key"] = build_key(aw["ocid"], aw["award_id"])

aw["award_amount"] = clean_amount(aw["award_amount"], "award_amount")
aw["award_currency"] = clean_text(aw["award_currency"], upper=True)
aw["award_date"] = clean_date(aw["award_date"])

awards = dedupe_table(aw, "award_key", "awards")
awards = awards[
    ["award_key", "ocid", "award_id", "award_amount", "award_currency", "award_date", "year"]
]
awards.dtypes

com_awards: 203,498 rows read
awards: 0 missing key, 0 duplicated -> 203,498 rows


award_key         string[python]
ocid              string[python]
award_id          string[python]
award_amount             float64
award_currency    string[python]
award_date        datetime64[ns]
year                       Int16
dtype: object

## awa_suppliers

In [5]:
sup = load_raw("com_awa_suppliers")

sup["ocid"] = clean_text(sup["ocid"])
for c in ["award_id", "supplier_id"]:
    sup[c] = clean_code(sup[c])
sup["supplier_name"] = clean_text(sup["supplier_name"], upper=True)

sup["award_key"] = build_key(sup["ocid"], sup["award_id"])
sup["supplier_key"] = build_key(sup["ocid"], sup["award_id"], sup["supplier_id"])

awa_suppliers = dedupe_table(sup, "supplier_key", "awa_suppliers")
awa_suppliers = awa_suppliers[
    ["supplier_key", "award_key", "ocid", "award_id", "supplier_id", "supplier_name", "year"]
]
awa_suppliers.dtypes

com_awa_suppliers: 199,581 rows read
awa_suppliers: 0 missing key, 0 duplicated -> 199,581 rows


supplier_key     string[python]
award_key        string[python]
ocid             string[python]
award_id         string[python]
supplier_id      string[python]
supplier_name    string[python]
year                      Int16
dtype: object

## contracts

In [6]:
con = load_raw("com_contracts")

con["ocid"] = clean_text(con["ocid"])
for c in ["contract_id", "award_id"]:
    con[c] = clean_code(con[c])
con["contract_title"] = clean_text(con["contract_title"])

con["contract_key"] = build_key(con["ocid"], con["contract_id"])
con["award_key"] = build_key(con["ocid"], con["award_id"])

for c in ["contract_amount", "final_value_amount"]:
    con[c] = clean_amount(con[c], c)
for c in ["date_signed", "period_start_date", "period_end_date"]:
    con[c] = clean_date(con[c])

# derived columns
con["period_days"] = (con["period_end_date"] - con["period_start_date"]).dt.days.astype("Int64")
con["invalid_period"] = con["period_days"].lt(0).fillna(False)
con["amount_variance"] = con["final_value_amount"] - con["contract_amount"]

contracts = dedupe_table(con, "contract_key", "contracts")
contracts = contracts[
    ["contract_key", "award_key", "ocid", "contract_id", "award_id", "contract_title",
     "contract_amount", "final_value_amount", "amount_variance", "date_signed",
     "period_start_date", "period_end_date", "period_days", "invalid_period", "year"]
]
contracts.dtypes

com_contracts: 196,827 rows read
contracts: 0 missing key, 0 duplicated -> 196,827 rows


contract_key          string[python]
award_key             string[python]
ocid                  string[python]
contract_id           string[python]
award_id              string[python]
contract_title        string[python]
contract_amount              float64
final_value_amount           float64
amount_variance              float64
date_signed           datetime64[ns]
period_start_date     datetime64[ns]
period_end_date       datetime64[ns]
period_days                    Int64
invalid_period               boolean
year                           Int16
dtype: object

## parties

In [7]:
par = load_raw("com_parties")

par["ocid"] = clean_text(par["ocid"])
for c in ["party_id", "identifier_id"]:
    par[c] = clean_code(par[c])
par["party_name"] = clean_text(par["party_name"], upper=True)
for c in ["identifier_scheme", "region", "department"]:
    par[c] = clean_text(par[c], upper=True)

par["party_key"] = build_key(par["ocid"], par["party_id"])

# roles arrive as a text-encoded list; collapse to a plain string plus one flag per role
roles = (
    clean_text(par["roles"])
    .str.replace(r"""[\[\]'\"]""", "", regex=True)
    .str.replace(r"\s*,\s*", ",", regex=True)
    .str.lower()
)
par["roles"] = roles
for r in ["buyer", "procuringentity", "supplier", "tenderer"]:
    par["is_" + r] = roles.str.contains(r, na=False)

# catalog mode: one row per party_id (most recent); ocid then points to a single process only
if PARTIES_BY_PARTY_ID:
    par = par[par["party_id"].notna()].sort_values("year").drop_duplicates("party_id", keep="last")

parties = dedupe_table(par, "party_key", "parties")
parties = parties[
    ["party_key", "ocid", "party_id", "party_name", "identifier_id", "identifier_scheme",
     "roles", "is_buyer", "is_procuringentity", "is_supplier", "is_tenderer",
     "region", "department", "year"]
]
parties.dtypes

com_parties: 3,143,082 rows read
parties: 0 missing key, 0 duplicated -> 3,143,082 rows


party_key             string[python]
ocid                  string[python]
party_id              string[python]
party_name            string[python]
identifier_id         string[python]
identifier_scheme     string[python]
roles                 string[python]
is_buyer                     boolean
is_procuringentity           boolean
is_supplier                  boolean
is_tenderer                  boolean
region                string[python]
department            string[python]
year                           Int16
dtype: object

## ten_tenderers

In [8]:
ten = load_raw("com_ten_tenderers")

ten["ocid"] = clean_text(ten["ocid"])
for c in ["tender_id", "tenderer_id"]:
    ten[c] = clean_code(ten[c])
ten["tenderer_name"] = clean_text(ten["tenderer_name"], upper=True)

ten["tenderer_key"] = build_key(ten["ocid"], ten["tenderer_id"])


ten_tenderers = dedupe_table(ten, "tenderer_key", "ten_tenderers")
ten_tenderers = ten_tenderers[
    ["tenderer_key", "ocid", "tender_id", "tenderer_id", "tenderer_name", "year"]
]
ten_tenderers.dtypes

com_ten_tenderers: 2,906,843 rows read
ten_tenderers: 0 missing key, 0 duplicated -> 2,906,843 rows


tenderer_key     string[python]
ocid             string[python]
tender_id        string[python]
tenderer_id      string[python]
tenderer_name    string[python]
year                      Int16
dtype: object

## Referential integrity

In [9]:
def check_orphans(child, fk, parent, pk, name):
    orphan = ~child[fk].isin(parent[pk])
    print(f"{name}: {orphan.sum():,} rows with no parent in {fk}")
    return child[~orphan] if DROP_ORPHANS else child


parties = check_orphans(parties, "ocid", records, "ocid", "parties")
ten_tenderers = check_orphans(ten_tenderers, "ocid", records, "ocid", "ten_tenderers")
awards = check_orphans(awards, "ocid", records, "ocid", "awards")
# awards goes first: contracts and suppliers are validated against what is left
contracts = check_orphans(contracts, "award_key", awards, "award_key", "contracts")
awa_suppliers = check_orphans(awa_suppliers, "award_key", awards, "award_key", "awa_suppliers")

parties: 0 rows with no parent in ocid
ten_tenderers: 0 rows with no parent in ocid
awards: 0 rows with no parent in ocid
contracts: 1,933 rows with no parent in award_key
awa_suppliers: 0 rows with no parent in award_key


In [10]:
# Investigation: contracts with no matching award in awards
print("Checked:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))

no_award = contracts[~contracts["award_key"].isin(awards["award_key"])]
n_processes = no_award["ocid"].nunique()


# 1) size of the problem
summary = pd.Series({
    "contracts_without_award": len(no_award),
    "pct_contracts": round(100 * len(no_award) / len(contracts), 2),
    "affected_processes": n_processes,
    "pct_processes_with_contracts": round(100 * n_processes / contracts["ocid"].nunique(), 2),
    "pct_amount": round(100 * no_award["contract_amount"].sum() / contracts["contract_amount"].sum(), 2),
})
print(summary.to_string())

# 2) concentration by process, and the awards that did load
top = (
    no_award.groupby("ocid").size().rename("contracts_without_award").to_frame()
    .join(awards.groupby("ocid").size().rename("loaded_awards"))
    .fillna({"loaded_awards": 0})
    .sort_values("contracts_without_award", ascending=False)
)
top["loaded_awards"] = top["loaded_awards"].astype(int)
top["cumulative_pct"] = (top["contracts_without_award"].cumsum() / top["contracts_without_award"].sum() * 100).round(1)
display(top.head(20))

# 3) examples: award_id requested by the contract vs. what actually exists in awards
loaded = awards.groupby("ocid")["award_id"].agg(lambda s: ", ".join(s))
examples = no_award[["ocid", "award_id"]].head(10).copy()
examples["loaded_awards"] = examples["ocid"].map(loaded)
display(examples)


# contracts whose award is missing from awards (kept; see quality note)
contracts = contracts.assign(
    award_in_awards=contracts["award_key"].isin(awards["award_key"])
)

Checked: 2026-10-03 23:16
contracts_without_award         1933.00
pct_contracts                      0.98
affected_processes               663.00
pct_processes_with_contracts       0.39
pct_amount                         0.30


,contracts_without_award,loaded_awards,cumulative_pct
ocid,,,
ocds-dgv273-seacev3-1072867,289,8,15.0
ocds-dgv273-seacev3-1076786,199,9,25.2
ocds-dgv273-seacev3-1072061,147,10,32.9
ocds-dgv273-seacev3-1075223,106,6,38.3
ocds-dgv273-seacev3-2024-2543-1416,48,1,40.8
ocds-dgv273-seacev3-2023-200404-5,43,1,43.0
ocds-dgv273-seacev3-1168067,29,1,44.5
ocds-dgv273-seacev3-1122087,26,1,45.9
ocds-dgv273-seacev3-2024-857-12,18,3,46.8


,ocid,award_id,loaded_awards
223,ocds-dgv273-seacev3-937629,937629-20609735601,937629-20530147551
641,ocds-dgv273-seacev3-2023-1068-31,936246-591232,"936246-591233, 936246-591234, 936246-591242, 9..."
642,ocds-dgv273-seacev3-2023-1068-31,936246-591232,"936246-591233, 936246-591234, 936246-591242, 9..."
643,ocds-dgv273-seacev3-2023-1068-31,936246-591239,"936246-591233, 936246-591234, 936246-591242, 9..."
644,ocds-dgv273-seacev3-2023-1068-31,936246-591239,"936246-591233, 936246-591234, 936246-591242, 9..."
646,ocds-dgv273-seacev3-2023-1068-31,936246-591240,"936246-591233, 936246-591234, 936246-591242, 9..."
647,ocds-dgv273-seacev3-2023-1068-31,936246-591240,"936246-591233, 936246-591234, 936246-591242, 9..."
2828,ocds-dgv273-seacev3-1076786,1076786-20601573416,"1076786-10176025315, 1076786-10091877045, 1076..."
2830,ocds-dgv273-seacev3-1076786,1076786-10436316122,"1076786-10176025315, 1076786-10091877045, 1076..."
2831,ocds-dgv273-seacev3-1076786,1076786-10400908848,"1076786-10176025315, 1076786-10091877045, 1076..."


### DATA QUALITY NOTE - Contracts with no matching award in `awards`

**Finding**
- 1,933 contracts (0.98% of 196,827) across 663 processes (0.39% of 171,709 processes with contracts) reference an `award_key` that doesn't exist in `awards`.
- Economic weight: 583.8M of 197,355.5M in `contract_amount` (0.30%). `contracts` has no currency field, so this figure is only indicative if currencies are mixed.
- Concentration: 4 processes account for 38% of the cases, and 20 processes account for 55%.

**Evidence this isn't a cleaning bug**
- `awards`: 203,498 rows read and 203,498 kept (0 missing key, 0 duplicates).
- The ids are well-formed (e.g. `937629-20609735601` vs. `937629-20530147551`).
- The referenced awards simply don't exist in the source CSVs.

**Cause**
- Inconsistency in the source data.

**Decision**
- Kept as-is (`DROP_ORPHANS = False`). The FK `contracts.award_key -> awards.award_key` is declared, and `PRAGMA foreign_key_check` lists the 1,933 cases.
- Reasons: these are real contracts; the monetary impact is only 0.3%; dropping them would bias the data against larger processes; filtering later is reversible, deleting is not.

**Impact on downstream analysis**
- These contracts have no supplier via `awa_suppliers`: supplier rankings will slightly underestimate them.
- When joining `contracts -> awards`, use a `LEFT JOIN`. An `INNER JOIN` silently drops them.

**How to identify or exclude them**
```sql
SELECT * FROM contracts c WHERE NOT EXISTS (SELECT 1 FROM awards a WHERE a.award_key = c.award_key);
```
- Or filter on `award_in_awards = 0`

In [11]:
tables = {
    "records": (records, "ocid"),
    "parties": (parties, "party_key"),
    "ten_tenderers": (ten_tenderers, "tenderer_key"),
    "awards": (awards, "award_key"),
    "contracts": (contracts, "contract_key"),
    "awa_suppliers": (awa_suppliers, "supplier_key"),
}

for name, (df, pk) in tables.items():
    assert df[pk].is_unique, f"{name}: {pk} has duplicates"

# percentage of nulls per column
for name, (df, _) in tables.items():
    nulls = (df.isna().mean() * 100).round(1)
    print(f"\n{name} ({len(df):,} rows)")
    print(nulls[nulls > 0].to_string() if (nulls > 0).any() else "no nulls")


records (236,234 rows)
budget_amount           27.3
procurement_method      14.6
tender_start_date        6.4
tender_end_date          6.4
number_of_tenderers      9.0
tender_duration_days     6.4

parties (3,143,082 rows)
region        92.5
department    92.5

ten_tenderers (2,906,843 rows)
no nulls

awards (203,498 rows)
no nulls

contracts (196,827 rows)
contract_title         0.2
final_value_amount    88.6
amount_variance       88.6

awa_suppliers (199,581 rows)
no nulls


## Load to SQLite

Dates are stored as text `YYYY-MM-DD HH:MM:SS` so `date()` and `strftime()` work in SQL, and flags as 0/1.

In [12]:
def prepare_for_sql(df):
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        elif pd.api.types.is_bool_dtype(out[c]):
            out[c] = out[c].astype("Int8")
    return out


PK = {
    "records": "ocid",
    "parties": "party_key",
    "ten_tenderers": "tenderer_key",
    "awards": "award_key",
    "contracts": "contract_key",
    "awa_suppliers": "supplier_key",
}
FK = {
    "parties": [("ocid", "records", "ocid")],
    "ten_tenderers": [("ocid", "records", "ocid")],
    "awards": [("ocid", "records", "ocid")],
    "contracts": [("award_key", "awards", "award_key")],
    "awa_suppliers": [("award_key", "awards", "award_key")],
}


def sql_type(series):
    if pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_float_dtype(series):
        return "REAL"
    return "TEXT"   # text and dates (already converted to text)


def create_table(conn, name, df):
    defs = []
    for c in df.columns:
        extra = " PRIMARY KEY NOT NULL" if c == PK[name] else ""
        defs.append(f'"{c}" {sql_type(df[c])}{extra}')
    for col, parent, parent_col in FK.get(name, []):
        defs.append(f'FOREIGN KEY ("{col}") REFERENCES "{parent}" ("{parent_col}")')
    conn.execute(f'CREATE TABLE "{name}" (\n  ' + ",\n  ".join(defs) + "\n)")


conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = OFF")   # during load; FKs are checked at the end

# drop child tables first
for t in ["awa_suppliers", "contracts", "ten_tenderers", "parties", "awards", "records"]:
    conn.execute(f'DROP TABLE IF EXISTS "{t}"')

# the order of `tables` already puts each parent before its children
for name, (df, _) in tables.items():
    data = prepare_for_sql(df)
    create_table(conn, name, data)
    data.to_sql(name, conn, if_exists="append", index=False)
    print(f"{name}: {len(data):,} rows loaded")

# indexes for FK joins (PKs already have their unique index)
for t, rel in FK.items():
    for col, _, _ in rel:
        conn.execute(f'CREATE INDEX IF NOT EXISTS ix_{t}_{col} ON "{t}"("{col}")')
conn.commit()

# rows that violate any FK
violations = conn.execute("PRAGMA foreign_key_check").fetchall()
print(f"\nrows violating a FK: {len(violations):,}")
conn.close()
print(f"Done. Database at: {DB_PATH}")

records: 236,234 rows loaded
parties: 3,143,082 rows loaded
ten_tenderers: 2,906,843 rows loaded
awards: 203,498 rows loaded
contracts: 196,827 rows loaded
awa_suppliers: 199,581 rows loaded

rows violating a FK: 1,933
Done. Database at: contracts.db


In [16]:
# Create SQL views for easier querying from Power BI and other clients

conn = sqlite3.connect(DB_PATH)
with open(os.path.join("Public Procurement Peru - Views.sql"), "r", encoding="utf-8") as f:
    conn.executescript(f.read())
conn.close()
print("Views created.")

Views created.
